<a href="https://colab.research.google.com/github/koblihable/text_comprehesion_llm/blob/main/llm_project_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# datasets gives us access to huggingface datasets library
!pip install -q transformers datasets diffusers langchain-community langchain_chroma langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00


In [ ]:
# imports
from google.colab import userdata
from huggingface_hub import login
import glob
import os
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.callbacks import StdOutCallbackHandler
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# sign in to Hugging Face
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
MODEL = "gpt-4o-mini"
# to store the vectors
vector_db = "vector_db"

In [ ]:
# Read in documents using LangChain's loaders
# Add doc type to the metadata
folders = glob.glob("/content/drive/MyDrive/Study/LLM project/library/*")

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

text_loader_kwargs = {'encoding': 'utf-8'}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])


In [ ]:
#files = glob.glob("/content/drive/MyDrive/Study/LLM project/library/*")

#def add_metadata(doc, doc_type):
#    doc.metadata["doc_type"] = doc_type
#    return doc
#
#text_loader_kwargs = {"encoding": "utf-8"}
#
#documents = []
#for f in files:
#    doc_type = os.path.basename(f)
#    loader = TextLoader(str(f))
#    #loader = DirectoryLoader(str(f), glob="**/*",loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
#    doc = loader.load()
#    print(doc)
#    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

In [ ]:
for doc in documents:
  print(doc.metadata["doc_type"])

bleak house
david copperfield
great expectations
oliver twist
the pickwick papers


In [ ]:
# create chunks for training
text_splitter = CharacterTextSplitter(chunk_size=250, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [ ]:
print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

Total number of chunks: 20562
Document types found: {'bleak house', 'oliver twist', 'david copperfield', 'great expectations', 'the pickwick papers'}


In [ ]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: bleak house, oliver twist, david copperfield, great expectations, the pickwick papers


In [ ]:
# embedding - mapping a chunk of text to a vector that represents the meaning
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

<ipython-input-10-9b1cc2768ecc>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# use chroma to crete the vector store
# don't load again if already exists
if os.path.exists(vector_db):
    Chroma(persist_directory=vector_db, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=vector_db)

In [ ]:
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 20562 documents


In [ ]:
# collection is a chroma object
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 20,562 vectors with 384 dimensions in the vector store


In [ ]:
# get the documents, metadata and the vectors from the collenction
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

In [ ]:
# turn the vectors into numpy arrays
vectors = np.array(result['embeddings'])

In [ ]:
documents = result['documents']
doc_types = [metadata['doc_type'] for metadata in result['metadatas']]
colors = [['blue', 'green', 'red', 'orange','yellow'][['bleak house', 'oliver twist', 'great expectations', 'the pickwick papers','david copperfield'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

In [ ]:
fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:


# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL, api_key=userdata.get('OPENAI_API_KEY'))

# Alternative - if you'd like to use Ollama locally, uncomment this line instead
#llm = ChatOpenAI(temperature=0.7, model_name='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama')

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [ ]:
query = "Who was mr Bumble"
result = conversation_chain.invoke({"question": query})
print(result["answer"])

Mr. Bumble is a character from Charles Dickens' novel "Oliver Twist." He is the beadle of the workhouse where Oliver is raised and is portrayed as a pompous and self-important man who enjoys exercising his authority over the paupers. He is depicted as a bully and has a cowardly nature, often deriving pleasure from petty cruelty.


In [ ]:
# set up a new conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

In [ ]:
# Let's investigate what gets sent behind the scenes
llm = ChatOpenAI(temperature=0.7, model_name=MODEL,api_key=userdata.get('OPENAI_API_KEY'))

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

retriever = vectorstore.as_retriever()

conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

query = "Who was mr Bumble?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Mr. Bumble, seeing with excruciating feelings, the delight of the two
old paupers, who were tittering together most rapturously, hesitated
for an instant. Mrs. Bumble, whose patience brooked no delay, caught up
a bowl of soap-suds, and motioning him towards the door, ordered him
instantly to depart, on pain of receiving the contents upon his portly
person.

Mr. Bumble had been despatched to make various preliminary inquiries,
with the view of finding out some captain or other who wanted a
cabin-boy without any friends; and was returning to the workhouse to
communicate the result of his mission; when he encountered at the gate,
no less a person th